# <font color="Green">**Notebook Purpose**</font>

This notebook generates a cohort-level treatment trajectory heatmaps that visualize medication usage patterns across semiannual time bins from 2019 to 2024. These heatmaps support descriptive analysis, cluster interpretation, and visual comparison of treatment trajectories. Optional exclusion parameters allow for selective removal of patients (e.g., for filtering outliers or evaluating subgroup patterns) as well the input of a custom ordering for design purposes.

---

### <font color="Red">Required Data</font>

The following preprocessed objects must be present in the working directory before running this notebook. Examples of how to create these data objects are available in other notebooks within the **Data Preprocessing** folder.


#### **1. `patient_bins.pkl`**  
A Python dictionary mapping each `patient_id` to a list of 12 prescription bins.

Format:  
- **Keys:** patient IDs  
- **Values:** list of 12 elements  
  - Each element is a list of prescription class strings  
    (e.g., `["MET"]`, `["SGLT2", "MET"]`)


#### **2. `sorted_cluster_patient_ids.pkl`**  
A dictionary mapping each cluster ID to an ordered list of patient IDs.

Format:  
- **Keys:** cluster IDs  
- **Values:** ordered list of patients belonging to that cluster  

This object must already be sorted, as the notebook uses the stored ordering directly for visualization.

* This data structure is different from clustering_patient_ids, which contains only a flat list of patient IDs from the cohort.


#### **3. (Optional) `excluded_patients` list**  
A Python list of patient IDs to remove from the heatmap.  
Useful for:  
- Excluding outliers  
- Removing incomplete trajectories  
- Producing subgroup-specific heatmaps


All objects must reference the same underlying patient cohort to ensure compatibility between cluster membership and prescription bins.


In [ ]:
import pandas as pd
import pickle
import numpy as np

from scipy.cluster.hierarchy import linkage, leaves_list

import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
import matplotlib.patches as mpatches

##<font color="black">**Read in Data**</font>

In [ ]:
with open('/content/patient_bins.pkl', 'rb') as f:
    patient_bins = pickle.load(f)

with open('/content/patient_vectors.pkl', 'rb') as f:
    patient_vectors = pickle.load(f)

with open('/content/sorted_cluster_patient_ids.pkl', 'rb') as f:
    cluster_patient_ids = pickle.load(f)

with open('/content/early_dropout_patients.pkl', 'rb') as f:
    early_dropout_patients = pickle.load(f)


##<font color="black">**Functions for Heatmap Creation**</font>

In [ ]:

# Prescription colors
PRESCRIPTION_COLORS = {
    'SUL': '#FFC0CB',       # Light pink
    'SGLT2': '#008000',     # Green
    'Insulin': '#FF0000',   # Red
    'MET': '#0000FF',       # Blue
    'DPP-4': '#8B4513',     # Brown
    'GLP-1': '#FFDB58',     # Mustard
    'GIP/GLP-1': '#40E0D0', # Turquoise
    'TZD': '#FF8C00',       # Bright Orange (DarkOrange)
    'Other': '#000000',     # Black
    'nothing': '#FFFFFF'    # White (no prescription)
}

# This function creates the mixing effect (e.g. MET (Blue) + Insulin (Red) --> Purple)
def mix_colors(colors):
    rgb_colors = np.array([to_rgb(PRESCRIPTION_COLORS[color]) for color in colors if color in PRESCRIPTION_COLORS])
    if len(rgb_colors) == 0:
        return np.array(to_rgb(PRESCRIPTION_COLORS['nothing']))
    mixed_rgb = np.mean(rgb_colors, axis=0)
    return np.array(mixed_rgb, dtype=np.float32)

def create_heatmap(patient_bins, cluster_patient_ids, title_str, cluster_ordering=None, excluded_patients=None):

    excluded = set(excluded_patients) if excluded_patients is not None else set()

    # Determine ordering of clusters
    if cluster_ordering is None:
        ordering = list(cluster_patient_ids.keys()) # Uses intrinsic ordering
    else:
        ordering = cluster_ordering

    # Build sorted patient list and cluster boundaries
    sorted_patients = []
    cluster_boundaries = []
    start_idx = 0

    for cluster_id in ordering:
        # Original patient list
        patient_list = cluster_patient_ids[cluster_id]

        # Remove excluded patients
        filtered_list = [pid for pid in patient_list if pid not in excluded]

        # Extend global list
        sorted_patients.extend(filtered_list)

        # Update and store new cluster boundary index
        start_idx += len(filtered_list)
        cluster_boundaries.append(start_idx)

    num_patients = len(sorted_patients)
    num_bins = 12  # 2019–2024 (2 semiannual bins per year)

    # Create heatmap array
    heatmap_data = np.ones((num_patients, num_bins, 3), dtype=np.float32)

    for i, patient_id in enumerate(sorted_patients):
        for j, prescriptions in enumerate(patient_bins[patient_id]):
            heatmap_data[i, j] = mix_colors(prescriptions)

    # Plot heatmap
    fig_size = (6, min(num_patients / 5, 10))
    fig, ax = plt.subplots(figsize=fig_size)
    ax.imshow(heatmap_data, aspect='auto')

    # Vertical lines between bins
    for bin_idx in range(1, num_bins):
        ax.axvline(x=bin_idx - 0.5, color='black', linewidth=1, linestyle='dashed')

    # Horizontal cluster boundaries (skip 0)
    for boundary in cluster_boundaries:
        if boundary > 0:
            ax.axhline(y=boundary - 0.5, color='black', linewidth=2)

    # Y-axis ticks
    y_ticks = range(500, num_patients + 1, 500)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([str(i) for i in y_ticks])
    ax.set_ylabel('Patient Count')

    # X-axis labels
    x_labels = [f'{year}-H{half}' for year in range(2019, 2025) for half in (1, 2)]
    ax.set_xticks(range(num_bins))
    ax.set_xticklabels(x_labels[:num_bins], rotation=45, ha='right', fontsize=10)

    # Legend
    patches = [
        mpatches.Patch(color=color, label=drug)
        for drug, color in PRESCRIPTION_COLORS.items()
    ]
    ax.legend(handles=patches, title="Prescription Representation",
              bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.title(title_str)
    return fig

##<font color="black">**Patient Medication Heatmap including the early dropout patients**</font>

In [ ]:
fig = create_heatmap(patient_bins, cluster_patient_ids, 'Patient Medication Heatmap')
plt.show()

##<font color="black">**Patient Medication Heatmap excluding the early dropout patients**</font>

In [ ]:
EDG_patient_ids = [pid for ids in early_dropout_patients.values() for pid in ids]
fig = create_heatmap(patient_bins, cluster_patient_ids, 'Patient Medication Heatmap (No EDG)', None, EDG_patient_ids)
plt.show()

##<font color="black">**Patient Medication Heatmap with Custom Ordering**</font>

The code block below orders clusters by similarity.

In [ ]:
cluster_centroids = []
cluster_ids = []

for cluster_id, patient_ids in cluster_patient_ids.items():

    vectors = vectors = [patient_vectors[pid] for pid in patient_ids]

    if vectors:
        centroid = np.mean(vectors, axis=0)
        cluster_centroids.append(centroid)
        cluster_ids.append(cluster_id)

cluster_centroids = np.array(cluster_centroids)

Z = linkage(cluster_centroids, method='ward')  # This produces a matrix that describes how similar each cluster is to the others
leaf_order = leaves_list(Z) # Produces a list of indices representing the optimal ordering of clusters based on similarity

ordered_cluster_ids = [cluster_ids[i] for i in leaf_order]

In [ ]:
fig = create_heatmap(patient_bins, cluster_patient_ids, 'Patient Medication Heatmap', ordered_cluster_ids, EDG_patient_ids)
plt.show()